<a href="https://colab.research.google.com/github/krishikasahni/Deep-Learning/blob/main/exp10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 128
epochs = 5
lr = 1e-3
num_classes = 10

transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_data, val_data, test_data = random_split(dataset, [train_size, val_size, test_size])
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size)

class SimpleViT(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(3*32*32, 512)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

# RESNET-18
resnet = models.resnet18(pretrained=False)
resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)
vit = SimpleViT()
vit.to(device)
resnet.to(device)

criterion = nn.CrossEntropyLoss()
opt_vit = optim.Adam(vit.parameters(), lr=lr)
opt_resnet = optim.Adam(resnet.parameters(), lr=lr)

def train(model, optimizer):
    model.train()
    total_loss = 0
    correct = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pred = out.argmax(dim=1)
        correct += (pred == y).sum().item()

    acc = correct / len(train_loader.dataset)
    return total_loss, acc

def test(model):
    model.eval()
    correct = 0

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()

    acc = correct / len(test_loader.dataset)
    return acc

for epoch in range(epochs):
    loss, acc = train(vit, opt_vit)
    print(f"ViT Epoch {epoch+1}: Loss={loss:.4f}, Train Acc={acc:.4f}")

vit_test_acc = test(vit)
print("ViT Test Accuracy:", vit_test_acc)

for epoch in range(epochs):
    loss, acc = train(resnet, opt_resnet)
    print(f"ResNet Epoch {epoch+1}: Loss={loss:.4f}, Train Acc={acc:.4f}")

resnet_test_acc = test(resnet)
print("ResNet Test Accuracy:", resnet_test_acc)

100%|██████████| 170M/170M [00:03<00:00, 45.8MB/s]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


ViT Epoch 1: Loss=566.2632, Train Acc=0.3640
ViT Epoch 2: Loss=514.1396, Train Acc=0.4229
ViT Epoch 3: Loss=496.3789, Train Acc=0.4472
ViT Epoch 4: Loss=484.7388, Train Acc=0.4587
ViT Epoch 5: Loss=471.8476, Train Acc=0.4729
ViT Test Accuracy: 0.4622
ResNet Epoch 1: Loss=496.8056, Train Acc=0.4204
ResNet Epoch 2: Loss=395.7813, Train Acc=0.5439
ResNet Epoch 3: Loss=345.9311, Train Acc=0.6032
ResNet Epoch 4: Loss=313.4131, Train Acc=0.6417
ResNet Epoch 5: Loss=289.0519, Train Acc=0.6729
ResNet Test Accuracy: 0.6214
